# Imports

In [1]:
import re
import optuna
import pickle

import numpy as np
import pandas as pd

from sklearn import set_config
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import StackingClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split, cross_val_predict

from feature_engine.encoding import RareLabelEncoder
from feature_engine.selection import DropFeatures

/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
set_config(enable_metadata_routing=True)

In [3]:
def dump_pickle(file_obj, file_path):
    with open(file_path, 'bw') as file:
        pickle.dump(file_obj, file)


def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)


def voting_cross_val_predict(probas: list[np.ndarray], weights: np.ndarray[float]) -> np.ndarray[float]:
    weights /= weights.sum()
    return np.sum([proba * weight for proba, weight in zip(probas, weights)], axis=0)

# Loading Dataset

In [4]:
X_train = pd.read_parquet('../data/processed/X_train_stacking.parquet')
y_train = pd.read_parquet('../data/processed/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test_stacking.parquet')

In [5]:
X_train.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,knn_proba
0,0.802607,0.891870,0.895659,0.883947,0.808497,0.887454
1,0.649405,0.831360,0.829854,0.779541,0.689888,0.519966
2,0.505904,0.851809,0.751522,0.758911,0.558520,0.209855
3,0.001713,0.005910,0.007057,0.007298,0.002275,0.000000
4,0.392205,0.565506,0.489033,0.482967,0.414144,0.297043


In [6]:
X_test.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,knn_proba
0,0.004591,0.031986,0.022774,0.020057,0.006419,0.000000
1,0.003032,0.011817,0.014957,0.018887,0.063872,0.117213
2,0.003777,0.011641,0.014307,0.011497,0.004517,0.000000
3,0.163131,0.544040,0.585898,0.485388,0.655870,0.099916
4,0.872172,0.967646,0.959451,0.959356,0.840450,1.000000


# Feature Engineering

In [7]:
X_train['voting_lgbm_cat_xgb_hist_proba'] = voting_cross_val_predict(
    [
        X_train['lgbm_proba'],
        X_train['cat_proba'],
        X_train['xgb_proba'],
        X_train['hist_proba']
    ], 
    weights=np.power([0.9484150827066593, 0.9481909576979181, 0.9477770044293032, 0.9485545192992528], 2)
)

In [8]:
X_test['voting_lgbm_cat_xgb_hist_proba'] = voting_cross_val_predict(
    [
        X_test['lgbm_proba'],
        X_test['cat_proba'],
        X_test['xgb_proba'],
        X_test['hist_proba']
    ], 
    weights=np.power([0.9484150827066593, 0.9481909576979181, 0.9477770044293032, 0.9485545192992528], 2)
)

# Machine Learning

In [25]:
def objective(trial, X, y):

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    aucs = []

    params = {
        "solver": trial.suggest_categorical("solver", ["saga"]),
        "C": trial.suggest_float("C", 1e-5, 100, log=True),
        "l1_ratio": trial.suggest_float("l1_ratio", 0.0, 1.0),
        "class_weight": trial.suggest_categorical("class_weight", [None, "balanced"]),
        "fit_intercept": trial.suggest_categorical("fit_intercept", [True, False]),
        "tol": trial.suggest_float("tol", 1e-6, 1e-2, log=True),
        "max_iter": trial.suggest_int("max_iter", 1000, 5000)
    }

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):

        X_train, X_valid = X.iloc[train_idx, :], X.iloc[valid_idx, :]
        y_train, y_valid = y.iloc[train_idx, 0], y.iloc[valid_idx, 0]

        model = LogisticRegression(**params)
        model.fit(X_train, y_train)

        proba = model.predict_proba(X_valid)[:, 1]

        auc = roc_auc_score(y_valid, proba)
        aucs.append(auc)

        trial.report(np.mean(aucs), step=fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(aucs)

features_to_use = ['lg_proba', 'knn_proba', 'voting_lgbm_cat_xgb_hist_proba']

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42), pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))
study.optimize(lambda trial: objective(trial, X_train[features_to_use], y_train), n_trials=500, n_jobs=-1, show_progress_bar=True)


print("Best trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-05-18 14:46:36,653] A new study created in memory with name: no-name-72a9c76d-7c28-4c51-804a-631b5f89331b
Best trial: 4. Best value: 0.5:   0%|          | 1/500 [00:02<19:27,  2.34s/it]

[I 2026-05-18 14:46:38,984] Trial 4 finished with value: 0.5 and parameters: {'solver': 'saga', 'C': 1.9166027703730564e-05, 'l1_ratio': 0.9378992291791106, 'class_weight': None, 'fit_intercept': False, 'tol': 0.0004149387921797278, 'max_iter': 4303}. Best is trial 4 with value: 0.5.


Best trial: 6. Best value: 0.904902:   0%|          | 2/500 [00:07<34:54,  4.21s/it]

[I 2026-05-18 14:46:44,506] Trial 6 finished with value: 0.9049024728241661 and parameters: {'solver': 'saga', 'C': 0.05418154752388287, 'l1_ratio': 0.4415492826714462, 'class_weight': 'balanced', 'fit_intercept': False, 'tol': 0.00802015487823809, 'max_iter': 1670}. Best is trial 6 with value: 0.9049024728241661.


Best trial: 2. Best value: 0.94962:   1%|          | 3/500 [00:08<21:26,  2.59s/it] 

...

[I 2026-05-18 14:53:58,306] Trial 483 finished with value: 0.9496323679229632 and parameters: {'solver': 'saga', 'C': 0.00016404646884609058, 'l1_ratio': 0.8079569954883311, 'class_weight': 'balanced', 'fit_intercept': False, 'tol': 2.319709986330781e-05, 'max_iter': 3641}. Best is trial 466 with value: 0.9496364649677398.


Best trial: 466. Best value: 0.949636: 100%|█████████▉| 499/500 [07:22<00:00,  1.12it/s]

[I 2026-05-18 14:53:59,387] Trial 499 pruned. 


Best trial: 466. Best value: 0.949636: 100%|██████████| 500/500 [07:24<00:00,  1.12it/s]

[I 2026-05-18 14:54:01,357] Trial 496 pruned. 
Best trial score:
0.9496364649677398

Best params:
{'solver': 'saga', 'C': 0.00013900263941740717, 'l1_ratio': 0.9094481647699592, 'class_weight': 'balanced', 'fit_intercept': False, 'tol': 0.00926265679750946, 'max_iter': 3517}


## Train Dataset

In [26]:
cv = StratifiedKFold(shuffle=True, random_state=42, n_splits=5)

stacking_proba = cross_val_predict(
    LogisticRegression(**study.best_params), 
    X_train[features_to_use], 
    y_train.PitNextLap, 
    cv=cv, 
    n_jobs=-1, 
    method='predict_proba'
)[:, 1]

In [27]:
X_train_stacking = X_train.copy()
X_train_stacking['stacking_lg_knn_voting_lgbm_cat_xgb_hist_proba'] = stacking_proba

# Test Dataset

In [28]:
stacking = LogisticRegression(**study.best_params).fit(X_train[features_to_use], y_train.PitNextLap)

In [29]:
X_test_stacking = X_test.copy()
X_test_stacking['stacking_lg_knn_voting_lgbm_cat_xgb_hist_proba'] = stacking.predict_proba(X_test[features_to_use])[:, 1]

# Saving

In [31]:
X_test_stacking.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,knn_proba,voting_lgbm_cat_xgb_hist_proba,stacking_lg_knn_voting_lgbm_cat_xgb_hist_proba
0,0.004591,0.031986,0.022774,0.020057,0.006419,0.000000,0.019849,0.509303
1,0.003032,0.011817,0.014957,0.018887,0.063872,0.117213,0.012173,0.505573
2,0.003777,0.011641,0.014307,0.011497,0.004517,0.000000,0.010304,0.504827
3,0.163131,0.544040,0.585898,0.485388,0.655870,0.099916,0.444558,0.696136
4,0.872172,0.967646,0.959451,0.959356,0.840450,1.000000,0.939648,0.852833


In [33]:
X_train_stacking.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,knn_proba,voting_lgbm_cat_xgb_hist_proba,stacking_lg_knn_voting_lgbm_cat_xgb_hist_proba
0,0.802607,0.891870,0.895659,0.883947,0.808497,0.887454,0.868510,0.826590
1,0.649405,0.831360,0.829854,0.779541,0.689888,0.519966,0.772514,0.801335
2,0.505904,0.851809,0.751522,0.758911,0.558520,0.209855,0.717012,0.782757
3,0.001713,0.005910,0.007057,0.007298,0.002275,0.000000,0.005494,0.502476
4,0.392205,0.565506,0.489033,0.482967,0.414144,0.297043,0.482416,0.703165


In [32]:
X_train_stacking.to_parquet('../data/processed/X_train_stacking2.parquet')
X_test_stacking.to_parquet('../data/processed/X_test_stacking2.parquet')